In [9]:
import os
import random
import warnings

import numpy as np
import pandas as pd

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    average_precision_score,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow.keras import layers, models

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


In [10]:
# Load processed time-series dataset
DATA_PATH = "processed_timeseries_level.csv"

df = pd.read_csv(DATA_PATH)
print("Raw shape:", df.shape)
print("Columns:", df.columns.tolist())

df.head()

Raw shape: (1514385, 10)
Columns: ['Device_ID', 'Device_Type', 'Timestamp', 'Simulated_Temperature_Variance', 'Simulated_Motor_Vibration_Hz', 'Simulated_Voltage_Drop', 'RUL_Hours', 'RUL_Days', 'RUL', 'Will_Fail_In_72_Hours']


,Device_ID,Device_Type,Timestamp,Simulated_Temperature_Variance,Simulated_Motor_Vibration_Hz,Simulated_Voltage_Drop,RUL_Hours,RUL_Days,RUL,Will_Fail_In_72_Hours
0,MD03449,Defibrillator,2025-04-11,1.034382,30.536111,0.690940,9999.0,416.625,416.625,0
1,MD03449,Defibrillator,2025-04-12,0.957793,31.524303,0.800019,9999.0,416.625,416.625,0
2,MD03449,Defibrillator,2025-04-13,1.049316,32.712801,0.713084,9999.0,416.625,416.625,0
3,MD03449,Defibrillator,2025-04-14,1.029413,32.247110,0.746806,9999.0,416.625,416.625,0
4,MD03449,Defibrillator,2025-04-15,0.946774,31.906548,0.738442,9999.0,416.625,416.625,0


In [11]:
# Keep only leakage-safe columns for LSTM training
required_cols = [
    "Device_ID",
    "Timestamp",
    "Simulated_Temperature_Variance",
    "Simulated_Motor_Vibration_Hz",
    "Simulated_Voltage_Drop",
    "Will_Fail_In_72_Hours",
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

model_df = df[required_cols].copy()
model_df["Timestamp"] = pd.to_datetime(model_df["Timestamp"])
model_df = model_df.sort_values(["Device_ID", "Timestamp"]).reset_index(drop=True)

feature_cols = [
    "Simulated_Temperature_Variance",
    "Simulated_Motor_Vibration_Hz",
    "Simulated_Voltage_Drop",
]
target_col = "Will_Fail_In_72_Hours"

print("Model shape:", model_df.shape)
print("Target distribution:\n", model_df[target_col].value_counts(normalize=True).rename("ratio"))

Model shape: (1514385, 6)
Target distribution:
 Will_Fail_In_72_Hours
0    0.978613
1    0.021387
Name: ratio, dtype: float64


In [12]:
# Device-aware split to avoid leakage across devices
unique_devices = model_df["Device_ID"].drop_duplicates().sample(frac=1, random_state=SEED).tolist()

n_devices = len(unique_devices)
train_end = int(0.70 * n_devices)
val_end = int(0.85 * n_devices)

train_devices = set(unique_devices[:train_end])
val_devices = set(unique_devices[train_end:val_end])
test_devices = set(unique_devices[val_end:])

train_df = model_df[model_df["Device_ID"].isin(train_devices)].copy()
val_df = model_df[model_df["Device_ID"].isin(val_devices)].copy()
test_df = model_df[model_df["Device_ID"].isin(test_devices)].copy()

print("Train devices:", len(train_devices), "| rows:", len(train_df))
print("Val devices:", len(val_devices), "| rows:", len(val_df))
print("Test devices:", len(test_devices), "| rows:", len(test_df))

for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    ratio = part[target_col].mean()
    print(f"{name} positive ratio: {ratio:.4f}")

Train devices: 2904 | rows: 1059960
Val devices: 622 | rows: 227030
Test devices: 623 | rows: 227395
train positive ratio: 0.0215
val positive ratio: 0.0208
test positive ratio: 0.0215


In [13]:
# Fit scaler on train only, then build sliding windows per device
WINDOW = 30
STRIDE = 3  # reduce overlap to speed up training

scaler = StandardScaler()
scaler.fit(train_df[feature_cols])

for part in (train_df, val_df, test_df):
    part[feature_cols] = scaler.transform(part[feature_cols])


def make_sequences(part_df, features, target, window=30, stride=1):
    X_list, y_list = [], []

    for _, grp in part_df.groupby("Device_ID", sort=False):
        grp = grp.sort_values("Timestamp")
        values = grp[features].to_numpy(dtype=np.float32)
        labels = grp[target].to_numpy(dtype=np.float32)

        if len(grp) < window:
            continue

        for i in range(0, len(grp) - window + 1, stride):
            end = i + window
            X_list.append(values[i:end])
            y_list.append(labels[end - 1])  # label at the window end

    if not X_list:
        return np.empty((0, window, len(features)), dtype=np.float32), np.empty((0,), dtype=np.float32)

    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32)


X_train, y_train = make_sequences(train_df, feature_cols, target_col, window=WINDOW, stride=STRIDE)
X_val, y_val = make_sequences(val_df, feature_cols, target_col, window=WINDOW, stride=STRIDE)
X_test, y_test = make_sequences(test_df, feature_cols, target_col, window=WINDOW, stride=STRIDE)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:", X_val.shape, "y_val:", y_val.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)
print("Train positives:", y_train.mean())

X_train: (325248, 30, 3) y_train: (325248,)
X_val: (69664, 30, 3) y_val: (69664,)
X_test: (69776, 30, 3) y_test: (69776,)
Train positives: 0.017890964


In [14]:
# Handle class imbalance with class weights
neg = float((y_train == 0).sum())
pos = float((y_train == 1).sum())

if pos == 0:
    raise ValueError("No positive samples in training set. Adjust split/window or regenerate data.")

weight_for_0 = (1.0 / neg) * (neg + pos) / 2.0
weight_for_1 = (1.0 / pos) * (neg + pos) / 2.0
class_weight = {0: weight_for_0, 1: weight_for_1}

print("Class weights:", class_weight)

BATCH_SIZE = 512

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(10000, seed=SEED).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

Class weights: {0: 0.5091084403732911, 1: 27.947069943289225}


In [15]:
# Build and train LSTM model
model = models.Sequential([
    layers.Input(shape=(WINDOW, len(feature_cols))),
    layers.LSTM(64, return_sequences=True),
    layers.Dropout(0.2),
    layers.LSTM(32),
    layers.Dropout(0.2),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.AUC(name="auc_roc"),
        tf.keras.metrics.AUC(name="auc_pr", curve="PR"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
    ],
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_auc_pr", mode="max", patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_auc_pr", mode="max", factor=0.5, patience=1, min_lr=1e-5),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1,
)

model.summary()

Epoch 1/8
636/636 ━━━━━━━━━━━━━━━━━━━━ 55s 80ms/step - auc_pr: 0.9067 - auc_roc: 0.9980 - loss: 0.0654 - precision: 0.3520 - recall: 0.9947 - val_auc_pr: 0.9272 - val_auc_roc: 0.9993 - val_loss: 0.0254 - val_precision: 0.7054 - val_recall: 1.0000 - learning_rate: 0.0010
Epoch 2/8
636/636 ━━━━━━━━━━━━━━━━━━━━ 52s 82ms/step - auc_pr: 0.9563 - auc_roc: 0.9995 - loss: 0.0129 - precision: 0.7774 - recall: 0.9981 - val_auc_pr: 0.9723 - val_auc_roc: 0.9997 - val_loss: 0.0150 - val_precision: 0.7831 - val_recall: 0.9992 - learning_rate: 0.0010
Epoch 3/8
636/636 ━━━━━━━━━━━━━━━━━━━━ 83s 83ms/step - auc_pr: 0.9618 - auc_roc: 0.9996 - loss: 0.0098 - precision: 0.8113 - recall: 0.9990 - val_auc_pr: 0.9383 - val_auc_roc: 0.9994 - val_loss: 0.0216 - val_precision: 0.7356 - val_recall: 1.0000 - learning_rate: 0.0010
Epoch 4/8
636/636 ━━━━━━━━━━━━━━━━━━━━ 54s 85ms/step - auc_pr: 0.9782 - auc_roc: 0.9998 - loss: 0.0082 - precision: 0.8242 - recall: 0.9990 - val_auc_pr: 0.9815 - val_auc_roc: 0.9998 - va

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 30, 64)         │        17,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 91,109 (355.90 KB)

 Trainable params: 30,369 (118.63 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 60,740 (237.27 KB)

In [16]:
# Evaluate with threshold tuning on validation PR curve
val_prob = model.predict(X_val, batch_size=1024, verbose=0).ravel()
test_prob = model.predict(X_test, batch_size=1024, verbose=0).ravel()

precision, recall, thresholds = precision_recall_curve(y_val, val_prob)
f1_scores = (2 * precision * recall) / (precision + recall + 1e-9)

# thresholds has length n-1 relative to precision/recall arrays
if len(thresholds) > 0:
    best_idx = int(np.nanargmax(f1_scores[:-1]))
    best_thr = float(thresholds[best_idx])
else:
    best_thr = 0.5

print(f"Best threshold from val F1: {best_thr:.4f}")

val_pred = (val_prob >= best_thr).astype(int)
test_pred = (test_prob >= best_thr).astype(int)

print("\nValidation metrics")
print("PR-AUC:", average_precision_score(y_val, val_prob))
print("ROC-AUC:", roc_auc_score(y_val, val_prob))
print(classification_report(y_val, val_pred, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_val, val_pred))

print("\nTest metrics")
print("PR-AUC:", average_precision_score(y_test, test_prob))
print("ROC-AUC:", roc_auc_score(y_test, test_prob))
print(classification_report(y_test, test_pred, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_test, test_pred))

Best threshold from val F1: 0.9897

Validation metrics
PR-AUC: 0.9982996127147484
ROC-AUC: 0.999970276390748
              precision    recall  f1-score   support

         0.0     0.9996    0.9998    0.9997     68479
         1.0     0.9897    0.9755    0.9826      1185

    accuracy                         0.9994     69664
   macro avg     0.9947    0.9877    0.9911     69664
weighted avg     0.9994    0.9994    0.9994     69664

Confusion matrix:
 [[68467    12]
 [   29  1156]]

Test metrics
PR-AUC: 0.9981553167844596
ROC-AUC: 0.9999666957086454
              precision    recall  f1-score   support

         0.0     0.9995    0.9997    0.9996     68554
         1.0     0.9850    0.9705    0.9777      1222

    accuracy                         0.9992     69776
   macro avg     0.9923    0.9851    0.9887     69776
weighted avg     0.9992    0.9992    0.9992     69776

Confusion matrix:
 [[68536    18]
 [   36  1186]]


In [17]:
# Save trained model and preprocessing metadata
os.makedirs("artifacts", exist_ok=True)

model.save("artifacts/lstm_failure_predictor.keras")
np.save("artifacts/feature_means.npy", scaler.mean_)
np.save("artifacts/feature_scales.npy", scaler.scale_)

meta = {
    "feature_cols": feature_cols,
    "window": WINDOW,
    "threshold": best_thr,
    "seed": SEED,
}
pd.Series(meta).to_json("artifacts/training_meta.json", indent=2)

print("Saved artifacts to ./artifacts")

Saved artifacts to ./artifacts
